# CS781 Phase 3 - cross-validation analysis: rare labels (CPU only)
The test leaderboard showed one reliable thing: **test contains Dissent, and our models under-predict it** (adding Dissent
votes to the ensembles gave +0.04 to +0.05). Dev has no Dissent, so it cannot guide this. Here the 80 TRAIN documents are
the validation set: every training document has an out-of-fold (OOF) prediction from a model that did not train on it.

- **Table A (H1, H2)** - do stronger rare-label weights (`w30`) or extra end-of-document ranges (`tail`) help? OOF macro-F1
  over all 13 labels, Dissent precision / recall / F1, dev F1, and a paired per-fold comparison with `base`.
- **Table B (H3)** - per-label decision biases (argmax of log p + b) fitted on OOF to maximise macro-F1; honest estimate
  by nested CV (fit on 4 folds, score the 5th); dev shows the effect on the other 11 labels.
- **Check** - does OOF reproduce the leaderboard finding (Dissent vote rule)? If yes, OOF is a valid stand-in for test.
- **Targeted submissions** (`cv01_...` to `cv06_...`), one hypothesis each.

**Inputs:** one dataset with the six `bundle_cv-*.zip` files (+ optionally the six earlier bundles, for the top10 transfer
test) + `cs781-p3-code` (latest). Internet ON, `HF_TOKEN`, **Accelerator: None**. Save & Run All (~5 min).

In [1]:
import os, sys, json, re, glob, shutil, subprocess, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs"
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))
# ---- runs from every bundle attached (zip files, or folders Kaggle already unzipped)
NAME_OK = re.compile(r"^([A-E]\d\d_\w+|[DE]cv_\w+_f\d)$")
os.makedirs(RUNS, exist_ok=True)
for z in glob.glob("/kaggle/input/**/bundle_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/summary.json", recursive=True):
    d = os.path.dirname(p); t = os.path.basename(d)
    if NAME_OK.match(t) and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
ALL = sorted(t for t in os.listdir(RUNS) if NAME_OK.match(t) and os.path.exists(f"{RUNS}/{t}/test_prob.npz"))
CV = [t for t in ALL if "cv_" in t]
OLD = [t for t in ALL if "cv_" not in t]
print(len(CV), "cross-validation runs;", len(OLD), "earlier runs (only needed for the top10 transfer test)")

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

40 cross-validation runs; 125 earlier runs (only needed for the top10 transfer test)


## Analysis (H1, H2, H3)

In [2]:
import numpy as np
import rr_data as D
import rr_eval as E
labels = D.load_vocab(f"{P3DATA}/label_vocab.json"); NL = len(labels); LIX = {l: i for i, l in enumerate(labels)}
DIS, OVR = LIX["Dissent"], LIX["PrecedentOverruled"]
gold, dom, split_ids = {}, {}, {}
for sp in ("train", "dev", "test"):
    split_ids[sp] = []
    for dm in D.DOMAINS:
        for x in D.load_split(D.split_path(P3DATA, dm, sp)):
            split_ids[sp].append(x["id"]); dom[x["id"]] = dm
            if sp != "test":
                gold[x["id"]] = np.array([LIX[l] for l in x["labels"]])
test_docs = [D.RRDoc(x["id"], D.DOMAINS.index(dm), x["text"]) for dm in D.DOMAINS
             for x in D.load_split(D.split_path(P3DATA, dm, "test"))]

_cache = {}
def probs(t, split):
    if (t, split) not in _cache:
        f = np.load(f"{RUNS}/{t}/{split}_prob.npz")
        _cache[(t, split)] = {str(k): f[f"p{i}"].astype(np.float32) for i, k in enumerate(f["ids"])}
    return _cache[(t, split)]
def mean_probs(dicts):
    return {k: np.mean([d[k] for d in dicts], axis=0) for k in dicts[0]}
def summ(t):
    return json.load(open(f"{RUNS}/{t}/summary.json"))

# ---- groups (model, variant) -> 5 fold runs
groups = {}
for t in CV:
    m, v, k = re.match(r"^([DE])cv_(\w+)_f(\d)$", t).groups()
    groups.setdefault((m, v), {})[int(k)] = t
FOLD = {}                                                      # train doc id -> fold
for g, runs in groups.items():
    for k, t in runs.items():
        for i in summ(t)["oof_ids"]:
            FOLD[i] = k
G = {}
for g, runs in sorted(groups.items()):
    oof = {}
    for t in runs.values():
        oof.update(probs(t, "oof"))
    G[g] = dict(runs=runs, oof=oof, dev=mean_probs([probs(t, "dev") for t in runs.values()]),
                test=mean_probs([probs(t, "test") for t in runs.values()]))
    miss = [i for i in split_ids["train"] if i not in oof]
    print(g, f"{len(runs)} folds", f"OOF covers {len(oof)}/{len(split_ids['train'])} train docs" + (f" (missing {len(miss)})" if miss else ""))

# ---- fast metric on stacked arrays (sklearn-exact: macro over all 13 labels, zero_division=0)
def stack(P, ids):
    ids = [i for i in ids if i in P]
    return np.concatenate([np.log(P[i] + 1e-9) for i in ids]), np.concatenate([gold[i] for i in ids]), ids
def per_label(pred, y):
    tp = np.bincount(y[pred == y], minlength=NL).astype(float)
    fp = np.bincount(pred, minlength=NL) - tp
    fn = np.bincount(y, minlength=NL) - tp
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return f1, p, r
def macro(X, y, b=None):
    return per_label((X if b is None else X + b).argmax(1), y)[0].mean()

# cross-check against the official-metric implementation
X, y, ids = stack(G[sorted(G)[0]]["oof"], split_ids["train"])
pred = {i: [labels[j] for j in G[sorted(G)[0]]["oof"][i].argmax(1)] for i in ids}
ref = E.evaluate({i: [labels[j] for j in gold[i]] for i in ids}, pred, labels, {i: dom[i] for i in ids})["macro_f1"]
assert abs(ref - macro(X, y)) < 1e-9, (ref, macro(X, y))

R = ["=== REPORT cross-validation (rare labels) ===",
     f"OOF = out-of-fold predictions on the {len(FOLD)} TRAIN documents (each predicted by a model that did not train on it).",
     f"Gold counts in train: Dissent={sum(int((gold[i] == DIS).sum()) for i in split_ids['train'])} sentences "
     f"in {sum(int((gold[i] == DIS).any()) for i in split_ids['train'])} docs; "
     f"PrecedentOverruled={sum(int((gold[i] == OVR).sum()) for i in split_ids['train'])} sentences.", ""]

# ---- H1 / H2: training variants, measured on OOF (paired by fold against base)
def fold_scores(P):
    out = []
    for k in range(5):
        X, y, _ = stack(P, [i for i in split_ids["train"] if FOLD.get(i) == k])
        out.append(macro(X, y))
    return np.array(out)
R.append("TABLE A - TRAINING VARIANTS (H1 class weights, H2 end-of-document ranges): OOF on train | dev (5-fold average model)")
R.append(f"{'model variant':16s} {'OOF F1':>7s} {'Dis P':>6s} {'Dis R':>6s} {'Dis F1':>6s} {'Ovr F1':>6s} {'dev F1':>7s}  "
         f"{'vs base: mean diff per fold, folds better':s}")
VAR_SCORE = {}
ORDER = ["base", "w30", "tail", "w30tail"]
for (m, v), g in sorted(G.items(), key=lambda kv: (kv[0][0], ORDER.index(kv[0][1]) if kv[0][1] in ORDER else 9)):
    X, y, _ = stack(g["oof"], split_ids["train"])
    f1, p, r = per_label(X.argmax(1), y)
    Xd, yd, _ = stack(g["dev"], split_ids["dev"])
    fs = fold_scores(g["oof"])
    g["fold_scores"] = fs
    cmp = ""
    if v != "base" and (m, "base") in G:
        d = fs - G[(m, "base")]["fold_scores"]
        cmp = f"{d.mean():+.4f}, {int((d > 0).sum())}/5"
    VAR_SCORE.setdefault(v, []).append(f1.mean())
    R.append(f"{m} {v:13s} {f1.mean():7.4f} {p[DIS]:6.3f} {r[DIS]:6.3f} {f1[DIS]:6.3f} {f1[OVR]:6.3f} {macro(Xd, yd):7.4f}  {cmp}")
BEST_V = max(VAR_SCORE, key=lambda v: np.mean(VAR_SCORE[v]))
R.append(f"-> best training variant on OOF (mean over D and E): {BEST_V}  "
         + ", ".join(f"{v} {np.mean(s):.4f}" for v, s in VAR_SCORE.items()))

# ---- H3: per-label decision biases fitted on OOF (argmax of log p + b), checked with nested CV and on dev
GRID = np.arange(-2.0, 4.01, 0.25)
MIN_GOLD = 20                                                  # labels rarer than this in OOF keep bias 0 (PrecedentOverruled)
def fit_bias(X, y, rounds=3):
    b = np.zeros(NL); best = macro(X, y, b)
    free = [j for j in range(NL) if (y == j).sum() >= MIN_GOLD]
    for _ in range(rounds):
        for j in free:
            for gval in GRID:
                c = b.copy(); c[j] = gval
                s = macro(X, y, c)
                if s > best + 1e-9:
                    best, b = s, c
    return b, best
def ens(keys, split):
    return mean_probs([G[k][split] for k in keys])
def h3(name, keys):
    oof, dev = ens(keys, "oof"), ens(keys, "dev")
    X, y, ids = stack(oof, split_ids["train"])
    b, fit_score = fit_bias(X, y)
    nested = []                                                # honest: fit on 4 folds, score the 5th
    for k in range(5):
        tr = np.concatenate([[FOLD[i] != k] * len(gold[i]) for i in ids]).astype(bool)
        bk, _ = fit_bias(X[tr], y[tr], rounds=2)
        nested.append((macro(X[~tr], y[~tr]), macro(X[~tr], y[~tr], bk)))
    nested = np.array(nested)
    Xd, yd, _ = stack(dev, split_ids["dev"])
    f1d0, _, _ = per_label(Xd.argmax(1), yd); f1d1, _, _ = per_label((Xd + b).argmax(1), yd)
    f10, _, r0 = per_label(X.argmax(1), y); f11, _, r1 = per_label((X + b).argmax(1), y)
    R.append(f"\n{name}: members {keys}")
    R.append(f"  OOF macro-F1 {macro(X, y):.4f} -> {fit_score:.4f} (in-sample);  nested CV (honest) "
             f"{nested[:, 0].mean():.4f} -> {nested[:, 1].mean():.4f}, better in {int((nested[:, 1] > nested[:, 0]).sum())}/5 folds")
    R.append(f"  dev macro-F1 {f1d0.mean():.4f} -> {f1d1.mean():.4f}  (dev has no Dissent/PrecOvr: shows only the cost/gain on the other 11)")
    R.append(f"  OOF Dissent F1 {f10[DIS]:.3f} -> {f11[DIS]:.3f} (recall {r0[DIS]:.3f} -> {r1[DIS]:.3f});  "
             f"PrecOvr F1 {f10[OVR]:.3f} -> {f11[OVR]:.3f}")
    R.append("  biases: " + " ".join(f"{labels[j][:8]}={b[j]:+.2f}" for j in range(NL) if b[j] != 0))
    R.append("  per-label OOF F1 change: " + " ".join(f"{labels[j][:8]}={f11[j] - f10[j]:+.2f}" for j in range(NL)
                                                   if abs(f11[j] - f10[j]) >= 0.005))
    return b, nested[:, 1].mean() - nested[:, 0].mean()

R.append("\nTABLE B - H3: PER-LABEL DECISION BIASES fitted on OOF")
KEYS_BASE = [k for k in [("D", "base"), ("E", "base")] if k in G]
KEYS_BEST = [k for k in [("D", BEST_V), ("E", BEST_V)] if k in G]
KEYS_ALL = sorted(G)
B_BASE, gain_base = h3("cv_base (D+E base)", KEYS_BASE)
B_BEST, gain_best = h3(f"cv_{BEST_V} (D+E {BEST_V})", KEYS_BEST) if BEST_V != "base" else (B_BASE, gain_base)
B_ALL, gain_all = h3("cv_all (every model x variant)", KEYS_ALL)

# ---- sanity check: does OOF reproduce the leaderboard finding (Dissent vote rule k=1 helps)?
oof_all = [G[k]["oof"] for k in KEYS_ALL]
X, y, ids = stack(ens(KEYS_ALL, "oof"), split_ids["train"])
votes = np.concatenate([np.sum([o[i].argmax(1) == DIS for o in oof_all], axis=0) for i in ids])
pv = np.where(votes >= 1, DIS, X.argmax(1))
R.append(f"\nCHECK - leaderboard rule 'Dissent if any member predicts it' on OOF (cv_all, {len(oof_all)} members): "
         f"macro-F1 {macro(X, y):.4f} -> {per_label(pv, y)[0].mean():.4f}  (on test it gave +0.044)")
print("\n".join(R))

('D', 'base') 5 folds OOF covers 80/80 train docs
('D', 'tail') 5 folds OOF covers 80/80 train docs
('D', 'w30') 5 folds OOF covers 80/80 train docs
('D', 'w30tail') 5 folds OOF covers 80/80 train docs
('E', 'base') 5 folds OOF covers 80/80 train docs
('E', 'tail') 5 folds OOF covers 80/80 train docs
('E', 'w30') 5 folds OOF covers 80/80 train docs
('E', 'w30tail') 5 folds OOF covers 80/80 train docs
=== REPORT cross-validation (rare labels) ===
OOF = out-of-fold predictions on the 80 TRAIN documents (each predicted by a model that did not train on it).
Gold counts in train: Dissent=284 sentences in 2 docs; PrecedentOverruled=3 sentences.

TABLE A - TRAINING VARIANTS (H1 class weights, H2 end-of-document ranges): OOF on train | dev (5-fold average model)
model variant     OOF F1  Dis P  Dis R Dis F1 Ovr F1  dev F1  vs base: mean diff per fold, folds better
D base           0.4942  0.000  0.000  0.000  0.000  0.5444  
D w30            0.5100  0.000  0.000  0.000  0.000  0.5200  +0.0127,

## Targeted submissions

In [3]:
# ---- targeted submissions: one hypothesis each
def old_top(n=10):
    ok = [t for t in OLD if summ(t)["dev"]["macro_f1"] >= 0.40]
    return sorted(ok, key=lambda t: -summ(t)["dev"]["macro_f1"])[:n]
def decide(P, b, voters, k):
    y = (np.log(P[k] + 1e-9) + b).argmax(1)
    if voters:                                                 # Dissent wherever any voter's argmax is Dissent
        v = np.sum([vp[k].argmax(1) == DIS for vp in voters], axis=0)
        y = np.where(v >= 1, DIS, y)
    return y
def write(name, test_P, b, dev_P, why, voters=None, dev_voters=None):
    pred = {k: [labels[j] for j in decide(test_P, b, voters, k)] for k in test_P}
    D.validate_submission(pred, test_docs, labels)
    p = f"{W}/_dev.json"
    json.dump({k: [labels[j] for j in decide(dev_P, b, dev_voters, k)] for k in dev_P}, open(p, "w"))
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {p} --labels {P3DATA}/label_vocab.json", shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    n = len(ROWS) + 1
    fn = f"cv{n:02d}_{name}.zip"
    with zipfile.ZipFile(f"{SUB}/{fn}", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(pred, ensure_ascii=False))
    cnt = {l: sum(v.count(l) for v in pred.values()) for l in labels}
    ROWS.append(dict(file=fn, dev=float(m.group(1)) if m else float("nan"), dissent=cnt["Dissent"],
                     prec_overruled=cnt["PrecedentOverruled"], why=why))

SUB = f"{W}/submissions"; shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB); ROWS = []
Z = np.zeros(NL)
write("cv_base", ens(KEYS_BASE, "test"), Z, ens(KEYS_BASE, "dev"),
      "control: 5-fold D+E LoRA ensemble, plain argmax (compare with ens_top10 0.5914)")
if BEST_V != "base":
    write(f"cv_{BEST_V}", ens(KEYS_BEST, "test"), Z, ens(KEYS_BEST, "dev"),
          f"H1/H2: best training variant on OOF ({BEST_V}) vs cv_base")
write(f"cv_{BEST_V}_bias", ens(KEYS_BEST, "test"), B_BEST, ens(KEYS_BEST, "dev"),
      "H3: + per-label biases fitted on OOF (vs the line above)")
write("cv_all_bias", ens(KEYS_ALL, "test"), B_ALL, ens(KEYS_ALL, "dev"),
      "H3 on the largest CV ensemble (all models x variants)")
top = old_top(10) if OLD else []
if len(top) >= 2:
    tP = mean_probs([probs(t, "test") for t in top]); dP = mean_probs([probs(t, "dev") for t in top])
    write("top10_bias", tP, B_BEST, dP,
          "H3 transfer: OOF biases applied to the earlier ens_top10 (compare 0.5914 plain, 0.6352 Dissent-vote)")
    allP = mean_probs([tP, ens(KEYS_ALL, "test")]); allD = mean_probs([dP, ens(KEYS_ALL, "dev")])
    write("top10+cv_all_bias", allP, B_ALL, allD, "everything combined: earlier top10 + all CV models, OOF biases")
    vt = [probs(t, "test") for t in top] + [G[k]["test"] for k in KEYS_ALL]
    vd = [probs(t, "dev") for t in top] + [G[k]["dev"] for k in KEYS_ALL]
    write("top10+cv_all_dissent1", allP, Z, allD, "best known recipe (top10 + Dissent vote, 0.6352) with the CV models added as "
          "members and voters", voters=vt, dev_voters=vd)
else:
    print("NOTE: the six earlier bundles (bundle_A ... bundle_extra) are not attached - top10 submissions skipped")

with open(f"{W}/SUBMISSIONS_cv.csv", "w") as f:
    f.write("file,dev_macro_f1,test_dissent,test_prec_overruled,hypothesis\n")
    for r in ROWS:
        f.write(f"{r['file']},{r['dev']:.4f},{r['dissent']},{r['prec_overruled']},\"{r['why']}\"\n")
with zipfile.ZipFile(f"{W}/cv_submissions.zip", "w") as z:
    for r in ROWS:
        z.write(f"{SUB}/{r['file']}", arcname=r["file"])
    z.write(f"{W}/SUBMISSIONS_cv.csv", arcname="SUBMISSIONS_cv.csv")
R2 = ["\nTARGETED SUBMISSIONS (submit in this order; dev = official evaluator, max 0.846)"]
for r in ROWS:
    R2.append(f"{r['file']:32s} dev {r['dev']:.4f}  Dissent={r['dissent']:4d} PrecOvr={r['prec_overruled']:3d}  {r['why']}")
print("\n".join(R2))
open(f"{W}/REPORT_cv.txt", "w").write("\n".join(R + R2))
print("\n=== END REPORT ===")


TARGETED SUBMISSIONS (submit in this order; dev = official evaluator, max 0.846)
cv01_cv_base.zip                 dev 0.5524  Dissent=   0 PrecOvr=  0  control: 5-fold D+E LoRA ensemble, plain argmax (compare with ens_top10 0.5914)
cv02_cv_tail.zip                 dev 0.5282  Dissent=   0 PrecOvr=  0  H1/H2: best training variant on OOF (tail) vs cv_base
cv03_cv_tail_bias.zip            dev 0.5050  Dissent=  47 PrecOvr=  0  H3: + per-label biases fitted on OOF (vs the line above)
cv04_cv_all_bias.zip             dev 0.5198  Dissent=  23 PrecOvr=  0  H3 on the largest CV ensemble (all models x variants)
cv05_top10_bias.zip              dev 0.5321  Dissent=  67 PrecOvr=  0  H3 transfer: OOF biases applied to the earlier ens_top10 (compare 0.5914 plain, 0.6352 Dissent-vote)
cv06_top10+cv_all_bias.zip       dev 0.5369  Dissent=  30 PrecOvr=  0  everything combined: earlier top10 + all CV models, OOF biases
cv07_top10+cv_all_dissent1.zip   dev 0.5516  Dissent=  31 PrecOvr=  0  best known r